# Module 1: ML Basics — Ejercicios Prácticos

**Objetivos:**
- Entender overfitting vs underfitting con código
- Visualizar bias-variance tradeoff
- Implementar validación cruzada correcta
- Aplicar regularización (L1, L2)

**Tiempo:** 30 min
**Dataset:** Synthetic (crearemos uno)

## Setup: Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## Ejercicio 1: Crear Dataset Sintético

In [ ]:
# Crear datos sintéticos no-lineales
np.random.seed(42)
n_samples = 100

# X: feature único
X = np.linspace(0, 10, n_samples).reshape(-1, 1)

# y: relación verdadera es cuadrática + ruido
y = 3 + 2*X.ravel() + X.ravel()**2 + np.random.normal(0, 10, n_samples)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} muestras")
print(f"Test set: {X_test.shape[0]} muestras")

# Visualizar
plt.figure(figsize=(10, 5))
plt.scatter(X_train, y_train, alpha=0.6, label='Train')
plt.scatter(X_test, y_test, alpha=0.6, label='Test')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Dataset Sintético (cuadrático + ruido)')
plt.show()

## Ejercicio 2: Overfitting vs Underfitting

Entrenamos modelos de diferentes complejidades:
- Grado 1: Línea recta (Underfitting)
- Grado 2: Parábola (Just right)
- Grado 10: Polinomio muy complejo (Overfitting)

In [ ]:
degrees = [1, 2, 3, 5, 10]
results = []

plt.figure(figsize=(15, 10))

for idx, degree in enumerate(degrees, 1):
    # Crear features polinómicas
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    # Entrenar modelo
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Evaluar
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    results.append({
        'degree': degree,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'gap': test_mse - train_mse
    })
    
    # Plotear
    plt.subplot(2, 3, idx)
    
    # Datos
    plt.scatter(X_train, y_train, alpha=0.5, label='Train')
    plt.scatter(X_test, y_test, alpha=0.5, label='Test')
    
    # Predicción (línea suave)
    X_smooth = np.linspace(0, 10, 300).reshape(-1, 1)
    X_smooth_poly = poly.transform(X_smooth)
    y_smooth = model.predict(X_smooth_poly)
    plt.plot(X_smooth, y_smooth, 'r-', linewidth=2, label='Modelo')
    
    plt.title(f'Degree {degree}\nTrain MSE: {train_mse:.1f}, Test MSE: {test_mse:.1f}')
    plt.xlabel('X')
    plt.ylabel('y')
    plt.legend(fontsize=8)
    plt.ylim(-50, 150)

plt.tight_layout()
plt.show()

# Tabla de resultados
df_results = pd.DataFrame(results)
print("\n📊 Resultados por Complejidad del Modelo:")
print(df_results.to_string(index=False))
print("\n💡 Análisis:")
print(f"- Degree 1: Train MSE alto → UNDERFITTING (modelo muy simple)")
print(f"- Degree 2: Train ≈ Test → OPTIMAL (balance)")
print(f"- Degree 10: Train bajo, Test alto → OVERFITTING (modelo memoriza)")

## Ejercicio 3: Visualizar Bias-Variance Tradeoff

In [ ]:
# Extraer train_mse vs test_mse
train_errors = [r['train_mse'] for r in results]
test_errors = [r['test_mse'] for r in results]
degrees_list = [r['degree'] for r in results]

plt.figure(figsize=(10, 6))
plt.plot(degrees_list, train_errors, 'o-', label='Train Error (Bias)', linewidth=2, markersize=8)
plt.plot(degrees_list, test_errors, 's-', label='Test Error (Bias + Variance)', linewidth=2, markersize=8)

# Marcar óptimo
optimal_idx = np.argmin(test_errors)
plt.axvline(x=degrees_list[optimal_idx], color='green', linestyle='--', alpha=0.5, label='Óptimo')

plt.xlabel('Complejidad del Modelo (Grado del Polinomio)', fontsize=12)
plt.ylabel('Error (MSE)', fontsize=12)
plt.title('Bias-Variance Tradeoff', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Anotaciones
plt.text(1, train_errors[0]+10, 'UNDERFITTING\n(Bias Alto)', fontsize=9, ha='center')
plt.text(10, test_errors[-1]-10, 'OVERFITTING\n(Variance Alta)', fontsize=9, ha='center')

plt.tight_layout()
plt.show()

## Ejercicio 4: Cross-Validation (K-Fold)

In [ ]:
# Usar degree 2 (óptimo)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# K-Fold CV
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = LinearRegression()

cv_scores = cross_val_score(model, X_poly, y, cv=kfold, scoring='neg_mean_squared_error')
cv_scores = -cv_scores  # Convertir a positivo

print("\n5-Fold Cross-Validation Results:")
print(f"Fold scores: {cv_scores}")
print(f"Mean CV score: {cv_scores.mean():.2f} ± {cv_scores.std():.2f}")

# Visualizar
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_scores, alpha=0.7, color='skyblue', edgecolor='black')
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.2f}')
plt.xlabel('Fold')
plt.ylabel('MSE')
plt.title('5-Fold Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print("\n💡 CV nos da confianza en que el modelo generaliza bien (scores similares en cada fold)")

## Ejercicio 5: Regularización (L1 vs L2)

In [ ]:
# Usar polinomio alto para ver efecto de regularización
poly = PolynomialFeatures(degree=10)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Normalizar features
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

# Prueba diferentes valores de regularización
alphas = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results = []
lasso_results = []

for alpha in alphas:
    # Ridge (L2)
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_poly_scaled, y_train)
    ridge_train_error = mean_squared_error(y_train, ridge.predict(X_train_poly_scaled))
    ridge_test_error = mean_squared_error(y_test, ridge.predict(X_test_poly_scaled))
    ridge_results.append((alpha, ridge_train_error, ridge_test_error))
    
    # Lasso (L1)
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_poly_scaled, y_train)
    lasso_train_error = mean_squared_error(y_train, lasso.predict(X_train_poly_scaled))
    lasso_test_error = mean_squared_error(y_test, lasso.predict(X_test_poly_scaled))
    lasso_results.append((alpha, lasso_train_error, lasso_test_error))

# Plotear
ridge_alphas, ridge_train, ridge_test = zip(*ridge_results)
lasso_alphas, lasso_train, lasso_test = zip(*lasso_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Ridge
ax1.plot(ridge_alphas, ridge_train, 'o-', label='Train Error', linewidth=2)
ax1.plot(ridge_alphas, ridge_test, 's-', label='Test Error', linewidth=2)
ax1.set_xlabel('Alpha (Regularización)')
ax1.set_ylabel('MSE')
ax1.set_title('Ridge (L2) Regularization')
ax1.set_xscale('log')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Lasso
ax2.plot(lasso_alphas, lasso_train, 'o-', label='Train Error', linewidth=2)
ax2.plot(lasso_alphas, lasso_test, 's-', label='Test Error', linewidth=2)
ax2.set_xlabel('Alpha (Regularización)')
ax2.set_ylabel('MSE')
ax2.set_title('Lasso (L1) Regularization')
ax2.set_xscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Resultados:")
print("- Sin regularización (alpha=0): Test error alto (overfitting)")
print("- Con regularización moderada: Test error baja (mejor generalización)")
print("- Con regularización excesiva: Train error sube mucho (underfitting)")
print("- Lasso tiende a zerear más features que Ridge")

## Resumen y Conclusiones

In [ ]:
print("="*60)
print("RESUMEN: ML BASICS")
print("="*60)
print()
print("1. OVERFITTING:")
print("   - Ocurre cuando el modelo memoriza datos")
print("   - Síntoma: Train error bajo, Test error alto")
print("   - Solución: Regularización, más datos, modelo más simple")
print()
print("2. BIAS-VARIANCE TRADEOFF:")
print("   - Bias: error sistemático (underfitting)")
print("   - Variance: sensibilidad al ruido (overfitting)")
print("   - Objetivo: encontrar equilibrio")
print()
print("3. TRAIN/VAL/TEST SPLIT:")
print("   - Train: aprende parámetros")
print("   - Val: ajusta hiperparámetros")
print("   - Test: evaluación honesta (nunca tocar)")
print()
print("4. CROSS-VALIDATION:")
print("   - Múltiples splits para validación robusta")
print("   - Reduce variabilidad debida a split aleatorio")
print()
print("5. REGULARIZACIÓN:")
print("   - L2 (Ridge): penaliza todos los pesos")
print("   - L1 (Lasso): zeroa features irrelevantes")
print("   - Elastic Net: combinación de ambas")
print()
print("="*60)